In [ ]:
%load_ext cash
%cash_on
%cash_badge print
%cash_debug on

In [ ]:
import os
import time
import urllib.request

In [ ]:
# Download GHCN-Daily data: by_year files and station metadata
data_dir = os.path.join(os.getcwd(), 'examples', 'large_scale_projects', 'data', 'noaa_weather')
os.makedirs(data_dir, exist_ok=True)

# GHCN-Daily by_year files (each ~100-200 MB compressed)
base_url = 'https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_year/'
years_to_download = [2022, 2023, 2024]

opener = urllib.request.build_opener()
opener.addheaders = [('User-Agent', 'Mozilla/5.0 (Cash-Benchmark/1.0; research project)')]
urllib.request.install_opener(opener)

t0 = time.time()

# Download station metadata
stations_url = 'https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt'
stations_path = os.path.join(data_dir, 'ghcnd-stations.txt')
if not os.path.exists(stations_path):
    print('Downloading station metadata...')
    urllib.request.urlretrieve(stations_url, stations_path)
    print(f'  Downloaded: {os.path.getsize(stations_path) / 1e6:.1f} MB')
else:
    print(f'Already downloaded: ghcnd-stations.txt ({os.path.getsize(stations_path) / 1e6:.1f} MB)')

# Download yearly data files
for year in years_to_download:
    fname = f'{year}.csv.gz'
    fpath = os.path.join(data_dir, fname)
    if os.path.exists(fpath):
        sz = os.path.getsize(fpath) / 1e6
        print(f'Already downloaded: {fname} ({sz:.1f} MB)')
    else:
        url = f'{base_url}{fname}'
        print(f'Downloading {fname}...')
        try:
            urllib.request.urlretrieve(url, fpath)
            sz = os.path.getsize(fpath) / 1e6
            elapsed = time.time() - t0
            print(f'  Downloaded: {fname} ({sz:.1f} MB, {elapsed:.0f}s)')
        except Exception as e:
            print(f'  FAILED: {e}')

elapsed = time.time() - t0
print(f'\nDownload complete in {elapsed:.0f}s')

# List data files
all_files = os.listdir(data_dir)
total_size = sum(os.path.getsize(os.path.join(data_dir, f)) for f in all_files)
print(f'Files: {all_files}')
print(f'Total data size: {total_size / 1e6:.0f} MB')

In [ ]:
# Load and parse station metadata (fixed-width format)
import pandas as _pd5
import os as _os5
import time as _time5

_data_dir = _os5.path.join(_os5.getcwd(), 'examples', 'large_scale_projects', 'data', 'noaa_weather')
_stations_path = _os5.path.join(_data_dir, 'ghcnd-stations.txt')

# GHCN-D stations file is fixed-width:
# ID: chars 1-11, Latitude: chars 13-20, Longitude: chars 22-30, Elevation: chars 32-37
# State: chars 39-40, Name: chars 42-71
_stations = _pd5.read_fwf(
    _stations_path,
    colspecs=[(0, 11), (12, 20), (21, 30), (31, 37), (38, 40), (41, 71)],
    names=['station_id', 'latitude', 'longitude', 'elevation', 'state', 'name'],
    dtype={'station_id': str, 'state': str, 'name': str}
)
print(f'Total stations: {len(_stations):,}')

# Focus on US stations (station IDs starting with 'US')
_us_stations = _stations[_stations['station_id'].str.startswith('US')].copy()
print(f'US stations: {len(_us_stations):,}')
print(f'States represented: {_us_stations["state"].nunique()}')
print(f'Elevation range: {_us_stations["elevation"].min():.0f}m to {_us_stations["elevation"].max():.0f}m')
print(f'\nSample stations:')
print(_us_stations.head(10).to_string(index=False))

In [ ]:
# Load GHCN-Daily observations for selected years
# Columns: STATION, DATE, ELEMENT, VALUE, M-FLAG, Q-FLAG, S-FLAG, OBS-TIME
# ELEMENT values we care about: TMAX, TMIN, PRCP, SNOW, SNWD
# VALUE: tenths of degrees C for temp, tenths of mm for precip

_years = [2022, 2023, 2024]
_col_names = ['station_id', 'date', 'element', 'value', 'mflag', 'qflag', 'sflag', 'obs_time']
_col_dtypes = {'station_id': str, 'date': str, 'element': str, 'mflag': str, 'qflag': str, 'sflag': str, 'obs_time': str}
_elements_keep = {'TMAX', 'TMIN', 'PRCP', 'SNOW', 'SNWD'}

_t0 = _time5.time()
_dfs = []
for _yr in _years:
    _fpath = _os5.path.join(_data_dir, f'{_yr}.csv.gz')
    if not _os5.path.exists(_fpath):
        print(f'Skipping {_yr}: file not found')
        continue
    _sz = _os5.path.getsize(_fpath) / 1e6
    print(f'Reading {_yr}.csv.gz ({_sz:.0f} MB)...')
    _df_yr = _pd5.read_csv(_fpath, names=_col_names, dtype=_col_dtypes,
                            usecols=['station_id', 'date', 'element', 'value'])
    # Filter to elements we care about and US stations
    _df_yr = _df_yr[_df_yr['element'].isin(_elements_keep)]
    _df_yr = _df_yr[_df_yr['station_id'].str.startswith('US')]
    _dfs.append(_df_yr)
    print(f'  {len(_df_yr):,} US records (filtered from full file)')

_obs = _pd5.concat(_dfs, ignore_index=True)
_obs['date'] = _pd5.to_datetime(_obs['date'], format='%Y%m%d')
_obs['year'] = _obs['date'].dt.year
_obs['month'] = _obs['date'].dt.month
_obs['day_of_year'] = _obs['date'].dt.dayofyear

_elapsed = _time5.time() - _t0
print(f'\nLoaded {len(_obs):,} observations in {_elapsed:.1f}s')
print(f'Memory: {_obs.memory_usage(deep=True).sum() / 1e6:.0f} MB')
print(f'Date range: {_obs["date"].min()} to {_obs["date"].max()}')
print(f'Unique stations: {_obs["station_id"].nunique():,}')
print(f'\nObservations by element:')
_element_counts = _obs['element'].value_counts()
for _elem in _element_counts.index:
    print(f'  {_elem}: {_element_counts[_elem]:,}')

In [ ]:
# ── Cell 6: Temperature analysis ──
# Convert tenths of degrees C to degrees F
# TMAX and TMIN are in tenths of degrees Celsius

_tmax = _obs[_obs['element'] == 'TMAX'].copy()
_tmax['temp_f'] = _tmax['value'] / 10.0 * 9/5 + 32
_tmin = _obs[_obs['element'] == 'TMIN'].copy()
_tmin['temp_f'] = _tmin['value'] / 10.0 * 9/5 + 32

print(f'TMAX records: {len(_tmax):,}')
print(f'TMIN records: {len(_tmin):,}')

# Merge station metadata for state info
_tmax = _tmax.merge(_us_stations[['station_id', 'state', 'name', 'latitude']], on='station_id', how='left')
_tmin = _tmin.merge(_us_stations[['station_id', 'state', 'name', 'latitude']], on='station_id', how='left')

# Monthly average max temperatures by state (2023)
_tmax_2023 = _tmax[_tmax['year'] == 2023]
_state_monthly_tmax = _tmax_2023.groupby(['state', 'month'])['temp_f'].mean().reset_index()
_state_monthly_tmax = _state_monthly_tmax.pivot(index='state', columns='month', values='temp_f')

# Hottest and coldest states by annual average max temp
_state_avg = _state_monthly_tmax.mean(axis=1).sort_values(ascending=False)
print(f'\nTop 10 hottest states (avg daily max temp, 2023):')
for _rank in range(min(10, len(_state_avg))):
    _st = _state_avg.index[_rank]
    _temp = _state_avg.iloc[_rank]
    print(f'  {_st}: {_temp:.1f}°F')

print(f'\nColdest 5 states:')
_cold5 = _state_avg.tail(5)
for _rank in range(len(_cold5)):
    _st = _cold5.index[_rank]
    _temp = _cold5.iloc[_rank]
    print(f'  {_st}: {_temp:.1f}°F')

In [ ]:
# ── Cell 7: Temperature trends chart ──
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# National daily average TMAX across all US stations, by year
_national_daily = _tmax.groupby(['year', 'day_of_year'])['temp_f'].mean().reset_index()

_fig1, _ax1 = plt.subplots(figsize=(14, 6))
_colors = {2022: 'steelblue', 2023: 'coral', 2024: 'seagreen'}
for _yr in _years:
    _yr_data = _national_daily[_national_daily['year'] == _yr]
    _ax1.plot(_yr_data['day_of_year'], _yr_data['temp_f'], 
             color=_colors.get(_yr, 'gray'), alpha=0.7, linewidth=1, label=str(_yr))

_ax1.set_xlabel('Day of Year')
_ax1.set_ylabel('Average Daily Max Temperature (°F)')
_ax1.set_title('US National Daily Max Temperature by Year')
_ax1.legend()
_ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('examples/large_scale_projects/data/noaa_weather/national_temp_trend.png', dpi=150)
plt.show()

# Year-over-year comparison
_annual_avg = _tmax.groupby('year')['temp_f'].mean()
print(f'\nAnnual average daily max temperature (US):')
for _yr in _years:
    if _yr in _annual_avg.index:
        print(f'  {_yr}: {_annual_avg[_yr]:.1f}°F')

In [ ]:
# ── Cell 8: Extreme temperature events ──
# Find days with extreme heat (TMAX >= 110°F / 43.3°C) and extreme cold (TMIN <= -30°F / -34.4°C)

_extreme_heat = _tmax[_tmax['temp_f'] >= 110].copy()
_extreme_cold = _tmin[_tmin['temp_f'] <= -30].copy()

print(f'Extreme heat events (TMAX >= 110°F): {len(_extreme_heat):,}')
print(f'Extreme cold events (TMIN <= -30°F): {len(_extreme_cold):,}')

# Extreme heat by year
print(f'\nExtreme heat by year:')
_heat_by_year = _extreme_heat.groupby('year').size()
for _yr in _years:
    _count = _heat_by_year.get(_yr, 0)
    print(f'  {_yr}: {_count:,} events')

# Top states for extreme heat
print(f'\nTop states for extreme heat (110°F+):')
_heat_by_state = _extreme_heat.groupby('state').size().sort_values(ascending=False)
for _rank in range(min(10, len(_heat_by_state))):
    _st = _heat_by_state.index[_rank]
    _count = _heat_by_state.iloc[_rank]
    print(f'  {_st}: {_count:,} events')

# Extreme cold by year
print(f'\nExtreme cold by year:')
_cold_by_year = _extreme_cold.groupby('year').size()
for _yr in _years:
    _count = _cold_by_year.get(_yr, 0)
    print(f'  {_yr}: {_count:,} events')

# Hottest single reading
if len(_extreme_heat) > 0:
    _hottest = _extreme_heat.loc[_extreme_heat['temp_f'].idxmax()]
    print(f'\nHottest reading: {_hottest["temp_f"]:.1f}°F at {_hottest["name"]} ({_hottest["state"]}) on {_hottest["date"].strftime("%Y-%m-%d")}')

if len(_extreme_cold) > 0:
    _coldest = _extreme_cold.loc[_extreme_cold['temp_f'].idxmin()]
    print(f'Coldest reading: {_coldest["temp_f"]:.1f}°F at {_coldest["name"]} ({_coldest["state"]}) on {_coldest["date"].strftime("%Y-%m-%d")}')

In [ ]:
# ── Cell 9: Precipitation analysis ──
# PRCP values are in tenths of mm

_prcp = _obs[_obs['element'] == 'PRCP'].copy()
_prcp['precip_in'] = _prcp['value'] / 10.0 / 25.4  # Convert tenths of mm to inches
_prcp = _prcp.merge(_us_stations[['station_id', 'state', 'name']], on='station_id', how='left')

print(f'Precipitation records: {len(_prcp):,}')

# Annual total precipitation by state (2023)
_prcp_2023 = _prcp[_prcp['year'] == 2023]
_station_annual = _prcp_2023.groupby(['station_id', 'state'])['precip_in'].sum().reset_index()
_state_precip = _station_annual.groupby('state')['precip_in'].median().sort_values(ascending=False)

print(f'\nWettest states (median station annual precip, 2023):')
for _rank in range(min(10, len(_state_precip))):
    _st = _state_precip.index[_rank]
    _precip = _state_precip.iloc[_rank]
    print(f'  {_st}: {_precip:.1f} inches')

print(f'\nDriest states:')
_dry5 = _state_precip.tail(5)
for _rank in range(len(_dry5)):
    _st = _dry5.index[_rank]
    _precip = _dry5.iloc[_rank]
    print(f'  {_st}: {_precip:.1f} inches')

# Extreme precipitation events (> 4 inches in a single day)
_heavy_rain = _prcp[_prcp['precip_in'] > 4.0]
print(f'\nHeavy rain events (>4 inches/day): {len(_heavy_rain):,}')
_heavy_by_year = _heavy_rain.groupby('year').size()
for _yr in _years:
    _count = _heavy_by_year.get(_yr, 0)
    print(f'  {_yr}: {_count:,} events')

In [ ]:
# ── Cell 10: Seasonal temperature chart by region ──

# Define US regions
_region_map = {
    'Northeast': ['CT', 'DE', 'ME', 'MD', 'MA', 'NH', 'NJ', 'NY', 'PA', 'RI', 'VT', 'DC'],
    'Southeast': ['AL', 'AR', 'FL', 'GA', 'KY', 'LA', 'MS', 'NC', 'SC', 'TN', 'VA', 'WV'],
    'Midwest': ['IL', 'IN', 'IA', 'KS', 'MI', 'MN', 'MO', 'NE', 'ND', 'OH', 'SD', 'WI'],
    'Southwest': ['AZ', 'NM', 'OK', 'TX'],
    'West': ['AK', 'CA', 'CO', 'HI', 'ID', 'MT', 'NV', 'OR', 'UT', 'WA', 'WY']
}

# Create reverse mapping: state -> region
_state_to_region = {}
for _region, _states in _region_map.items():
    for _st in _states:
        _state_to_region[_st] = _region

_tmax['region'] = _tmax['state'].map(_state_to_region)

# Monthly avg by region (2023)
_tmax_2023_r = _tmax[_tmax['year'] == 2023]
_regional_monthly = _tmax_2023_r.groupby(['region', 'month'])['temp_f'].mean().reset_index()

_fig2, _ax2 = plt.subplots(figsize=(12, 6))
_region_colors = {'Northeast': 'steelblue', 'Southeast': 'coral', 'Midwest': 'seagreen',
                   'Southwest': 'goldenrod', 'West': 'mediumpurple'}
_month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

for _region in ['Northeast', 'Southeast', 'Midwest', 'Southwest', 'West']:
    _rdata = _regional_monthly[_regional_monthly['region'] == _region]
    _ax2.plot(_rdata['month'], _rdata['temp_f'], 'o-', 
             color=_region_colors[_region], linewidth=2, label=_region, markersize=6)

_ax2.set_xlabel('Month')
_ax2.set_ylabel('Average Daily Max Temperature (°F)')
_ax2.set_title('Monthly Temperature by US Region (2023)')
_ax2.set_xticks(range(1, 13))
_ax2.set_xticklabels(_month_names)
_ax2.legend()
_ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('examples/large_scale_projects/data/noaa_weather/regional_temp.png', dpi=150)
plt.show()

# Regional summary
print(f'Regional average max temperature (2023):')
for _region in ['Northeast', 'Southeast', 'Midwest', 'Southwest', 'West']:
    _rdata = _regional_monthly[_regional_monthly['region'] == _region]
    _jul = _rdata[_rdata['month'] == 7]['temp_f'].values
    _jan = _rdata[_rdata['month'] == 1]['temp_f'].values
    _jul_t = _jul[0] if len(_jul) > 0 else 0
    _jan_t = _jan[0] if len(_jan) > 0 else 0
    print(f'  {_region:12s}: Jan={_jan_t:.0f}°F, Jul={_jul_t:.0f}°F, range={_jul_t - _jan_t:.0f}°F')

In [ ]:
# ── Cell 11: Snow analysis ──

_snow = _obs[_obs['element'] == 'SNOW'].copy()
_snow['snow_in'] = _snow['value'] / 25.4  # tenths of mm to inches  
_snow = _snow.merge(_us_stations[['station_id', 'state', 'name']], on='station_id', how='left')

print(f'Snow records: {len(_snow):,}')

# Annual total snowfall by state
_snow_positive = _snow[_snow['snow_in'] > 0]
_station_snow = _snow_positive.groupby(['station_id', 'state', 'year'])['snow_in'].sum().reset_index()
_state_snow_annual = _station_snow.groupby(['state', 'year'])['snow_in'].median().reset_index()

# Snowiest states (median station annual snowfall, averaged across years)
_state_snow_avg = _state_snow_annual.groupby('state')['snow_in'].mean().sort_values(ascending=False)
print(f'\nSnowiest states (median station annual snowfall):')
for _rank in range(min(10, len(_state_snow_avg))):
    _st = _state_snow_avg.index[_rank]
    _snow_val = _state_snow_avg.iloc[_rank]
    print(f'  {_st}: {_snow_val:.1f} inches')

# Biggest single-day snowfall events
_big_snow = _snow[_snow['snow_in'] >= 12].sort_values('snow_in', ascending=False)
print(f'\nBig snowfall events (>=12 inches/day): {len(_big_snow):,}')
if len(_big_snow) > 0:
    print(f'\nTop 10 biggest single-day snowfalls:')
    _top_snow = _big_snow.head(10)
    for _rank in range(len(_top_snow)):
        _row = _top_snow.iloc[_rank]
        print(f'  {_row["snow_in"]:.1f}" at {_row["name"]} ({_row["state"]}) on {_row["date"].strftime("%Y-%m-%d")}')

In [ ]:
# ── Cell 12: Summary statistics ──
print('=' * 60)
print('PROJECT 5: NOAA Weather Station Analysis - Summary')
print('=' * 60)
print(f'\nDataset: GHCN-Daily {min(_years)}-{max(_years)}')
print(f'Total observations: {len(_obs):,}')
print(f'US stations: {_obs["station_id"].nunique():,}')
print(f'Memory: {_obs.memory_usage(deep=True).sum() / 1e6:.0f} MB')
print(f'Load time: {_elapsed:.1f}s')
print(f'\nTemperature Records:')
print(f'  TMAX: {len(_tmax):,}, TMIN: {len(_tmin):,}')
print(f'  Extreme heat events (110°F+): {len(_extreme_heat):,}')
print(f'  Extreme cold events (-30°F-): {len(_extreme_cold):,}')
print(f'\nPrecipitation Records: {len(_prcp):,}')
print(f'  Heavy rain events (>4 in/day): {len(_heavy_rain):,}')
print(f'\nSnow Records: {len(_snow):,}')
print(f'  Big snow events (>=12 in/day): {len(_big_snow):,}')
print(f'\nHottest state (2023): {_state_avg.index[0]} ({_state_avg.iloc[0]:.1f}°F avg daily max)')
print(f'Coldest state (2023): {_state_avg.index[-1]} ({_state_avg.iloc[-1]:.1f}°F avg daily max)')
print('=' * 60)